# Stock Price Prediction

## Problem Statement
Stock price prediction is a complex task due to the high volatility and randomness of financial markets. Traditional statistical models often fail to capture long-term dependencies in stock price movements. Deep learning, particularly Long Short-Term Memory (LSTM) networks, has shown great potential in handling time-series forecasting by learning from historical patterns.

## Step-by-Step

## Precheck

In [1]:
import warnings

warnings.filterwarnings("ignore")

In [2]:
%%time
%pip install -qU ipykernel ipython-autotime ipywidgets pip-system-certs  --force-reinstall --break-system-packages --no-warn-script-location 

%load_ext autotime

error: uninstall-no-record-file

× Cannot uninstall pip 25.2
╰─> The package's contents are unknown: no RECORD file was found for pip.

hint: The package was installed by brew. You should check if it can uninstall the package.
Note: you may need to restart the kernel to use updated packages.
CPU times: user 23.5 ms, sys: 15.9 ms, total: 39.4 ms
Wall time: 4.16 s
time: 255 μs (started: 2025-12-23 16:30:03 -08:00)


In [3]:
%%time
!python3 -V && pip3 -V

Python 3.13.7
pip 25.3 from /opt/homebrew/lib/python3.12/site-packages/pip (python 3.12)
CPU times: user 2.12 ms, sys: 4.93 ms, total: 7.05 ms
Wall time: 443 ms
time: 443 ms (started: 2025-12-23 16:30:03 -08:00)


### Install dependencies

In [4]:
!pip3 install -qU yfinance torch numpy pandas scikit-learn matplotlib certifi urllib3 requests

time: 1.37 s (started: 2025-12-23 16:30:03 -08:00)


### Load & Preprocess Stock Data

In [5]:
# Fix SSL certificate problem: unable to get local issuer certificate
import os
import ssl
import certifi
import platform

# Get the path to certifi's certificate bundle
cert_path = certifi.where()
print(f"Using SSL certificates from: {cert_path}")

# Method 1: Set environment variables for Python SSL and curl
os.environ["SSL_CERT_FILE"] = cert_path
os.environ["REQUESTS_CA_BUNDLE"] = cert_path
# For curl_cffi (used by yfinance) - these are the key variables
os.environ["CURL_CA_BUNDLE"] = cert_path
os.environ["CURL_CAINFO"] = cert_path
# Additional curl environment variables
os.environ["CURLOPT_CAINFO"] = cert_path
os.environ["CURLOPT_CAPATH"] = os.path.dirname(cert_path)

# On macOS, also try to use system certificates if available
if platform.system() == "Darwin":
    # macOS typically has certificates in these locations
    mac_cert_paths = [
        "/etc/ssl/cert.pem",
        "/usr/local/etc/openssl/cert.pem",
        "/opt/homebrew/etc/openssl/cert.pem",
    ]
    for mac_path in mac_cert_paths:
        if os.path.exists(mac_path):
            os.environ["CURL_CA_BUNDLE"] = mac_path
            os.environ["CURL_CAINFO"] = mac_path
            print(f"Also found macOS certificates at: {mac_path}")
            break

# Method 2: Configure Python SSL context to use certifi certificates
ssl_context = ssl.create_default_context(cafile=cert_path)
ssl._create_default_https_context = lambda: ssl_context

# Method 3: For requests library
import urllib3

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

# Method 4: Patch curl_cffi Session to disable SSL verification by default
# This is the most reliable fix for yfinance SSL issues
try:
    from curl_cffi import requests as curl_requests

    # Store original request method
    original_request = curl_requests.Session.request

    # Create patched request method that disables SSL verification
    def patched_request(self, method, url, **kwargs):
        # Always disable SSL verification
        kwargs["verify"] = False
        return original_request(self, method, url, **kwargs)

    # Apply the patch
    curl_requests.Session.request = patched_request
    print("✓ Patched curl_cffi Session to disable SSL verification")
    print(
        "⚠️  WARNING: SSL verification is disabled. This is less secure but necessary for yfinance to work."
    )
except (ImportError, AttributeError) as e:
    print(f"Could not patch curl_cffi: {e}")
    print("Will try environment variables only")

print("SSL certificate configuration completed!")

Using SSL certificates from: /opt/homebrew/lib/python3.12/site-packages/certifi/cacert.pem
Also found macOS certificates at: /etc/ssl/cert.pem
✓ Patched curl_cffi Session to disable SSL verification
⚠️  WARNING: SSL verification is disabled. This is less secure but necessary for yfinance to work.
SSL certificate configuration completed!
time: 41.1 ms (started: 2025-12-23 16:30:05 -08:00)


#### Alternative SSL Fix (Run this cell if SSL errors persist)

If you still encounter SSL certificate errors after running the previous cell, you may need to:
1. Restart the kernel and run all cells from the beginning
2. Update your system's CA certificates
3. Use the fallback method in the data download cell (which disables SSL verification)


#### Define variables

In [6]:
import yfinance as yf
import pandas as pd

# Configure yfinance (use new API instead of deprecated enable_debug_mode)
yf.config.debug.logging = False  # Set to True for debug logs
yf.set_tz_cache_location("./cache/")

# Download stock data
stock = "MSFT"
startDate = "1985-01-01"
todaysDate = pd.Timestamp.today().date().strftime("%Y-%m-%d")

print(f"Downloading stock data for '{stock}' from {startDate} to '{todaysDate}' ")

try:
    # Download stock data using yfinance
    ticker = yf.Ticker(stock)
    data = ticker.history(period="max", start=startDate, end=todaysDate)

    # Verify data was downloaded successfully
    if data.empty:
        print("Warning: No data downloaded. Trying alternative method...")
        # Alternative: use yf.download
        data = yf.download(stock, start=startDate, end=todaysDate, progress=False)
        if not data.empty:
            print(f"Successfully downloaded {len(data)} rows using alternative method")
    else:
        print(f"Successfully downloaded {len(data)} rows of data")

    print(f"\nData shape: {data.shape}")
    print(f"\nFirst few rows:")
    print(data.head())
    print(f"\nLast few rows:")
    print(data.tail())

except Exception as e:
    print(f"Error downloading data: {e}")
    print("\nTrying with SSL verification disabled as fallback...")
    # Fallback: disable SSL verification (less secure but works)
    # This is a workaround for SSL certificate issues
    import warnings

    warnings.filterwarnings("ignore", message="Unverified HTTPS request")

    # Configure yfinance to skip SSL verification
    # Note: This is less secure but may be necessary in some environments
    try:
        # Try to configure curl_cffi to skip verification
        os.environ["CURLOPT_SSL_VERIFYPEER"] = "0"
        os.environ["CURLOPT_SSL_VERIFYHOST"] = "0"

        # Retry with SSL verification disabled
        ticker = yf.Ticker(stock)
        data = ticker.history(period="max", start=startDate, end=todaysDate)

        if not data.empty:
            print(
                f"Successfully downloaded {len(data)} rows (with SSL verification disabled)"
            )
        else:
            raise Exception("Data is still empty after disabling SSL verification")
    except Exception as e2:
        print(f"Still failed: {e2}")
        print(
            "\nPlease check your internet connection and SSL certificate configuration."
        )
        raise

data

Successfully downloaded 10023 rows of data

Data shape: (10023, 7)

First few rows:
                               Open      High       Low     Close      Volume  \
Date                                                                            
1986-03-13 00:00:00-05:00  0.054086  0.062040  0.054086  0.059389  1031788800   
1986-03-14 00:00:00-05:00  0.059389  0.062571  0.059389  0.061510   308160000   
1986-03-17 00:00:00-05:00  0.061510  0.063101  0.061510  0.062571   133171200   
1986-03-18 00:00:00-05:00  0.062571  0.063101  0.060449  0.060979    67766400   
1986-03-19 00:00:00-05:00  0.060979  0.061510  0.059389  0.059919    47894400   

                           Dividends  Stock Splits  
Date                                                
1986-03-13 00:00:00-05:00        0.0           0.0  
1986-03-14 00:00:00-05:00        0.0           0.0  
1986-03-17 00:00:00-05:00        0.0           0.0  
1986-03-18 00:00:00-05:00        0.0           0.0  
1986-03-19 00:00:00-05:00     

,Open,High,Low,Close,Volume,Dividends,Stock Splits
Date,,,,,,,
1986-03-13 00:00:00-05:00,0.054086,0.062040,0.054086,0.059389,1031788800,0.0,0.0
1986-03-14 00:00:00-05:00,0.059389,0.062571,0.059389,0.061510,308160000,0.0,0.0
1986-03-17 00:00:00-05:00,0.061510,0.063101,0.061510,0.062571,133171200,0.0,0.0
1986-03-18 00:00:00-05:00,0.062571,0.063101,0.060449,0.060979,67766400,0.0,0.0
1986-03-19 00:00:00-05:00,0.060979,0.061510,0.059389,0.059919,47894400,0.0,0.0
...,...,...,...,...,...,...,...
2025-12-16 00:00:00-05:00,471.910004,477.890015,470.880005,476.390015,20705600,0.0,0.0
2025-12-17 00:00:00-05:00,476.910004,480.000000,475.000000,476.119995,24527200,0.0,0.0
2025-12-18 00:00:00-05:00,478.190002,489.600006,477.890015,483.980011,28573500,0.0,0.0


time: 1.4 s (started: 2025-12-23 16:30:05 -08:00)


In [8]:
import torch
import numpy as np
from sklearn.preprocessing import MinMaxScaler


# Extract 'Close' prices and scale them
scaler = MinMaxScaler()
data_scaled = scaler.fit_transform(data[["Close"]])


# Create time-series sequences
def create_sequences(data, seq_length=50):
    X, y = [], []
    for i in range(len(data) - seq_length):
        X.append(data[i : i + seq_length])
        y.append(data[i + seq_length])
    return np.array(X), np.array(y)


seq_length = 50  # Lookback period
X, y = create_sequences(data_scaled, seq_length)

# Train-test split (80-20)
train_size = int(0.8 * len(X))
X_train, X_test = X[:train_size], X[train_size:]
y_train, y_test = y[:train_size], y[train_size:]

# Convert to PyTorch tensors
X_train_tensor, X_test_tensor = torch.Tensor(X_train), torch.Tensor(X_test)
y_train_tensor, y_test_tensor = torch.Tensor(y_train), torch.Tensor(y_test)

time: 2.03 s (started: 2025-12-23 16:31:39 -08:00)
